# Rung 3 — Round-0.5 LABELER fine-tune (real strips only)

Throwaway checkpoint for the strip-label **emitter** (`docs/RUNG3.md` §1a.5) — **never shipped**, no
browser gate, never copied to `apps/web/public/models/`. It exists so the notaarsivleri emit (and the
`MEASURES_PER_STRIP=2` re-slice emit) runs on a model that has seen real pages: better row alignment
(yield), less nd-gate review labor, and a signature vote that stops unanimously misreading hicaz.

- **Start weights:** `rung22-stemfix-best` (fine-tune FROM it — forgetting synthetic is fine, the
  labeler only ever decodes real pages).
- **Data:** `data/real/rung3/strips_r1` — the promoted, human-adjudicated pool (418 strips / 48 pieces,
  every label verdicted ok/fix by hand). Exam pieces are excluded BY CONSTRUCTION (the emitter never
  emitted them into this dir); val = 8 held-out pieces / 56 strips (`split.json`).
- **Recipe:** short + small LR — `--lr 1e-5 --max-steps 1200 --warmup-steps 50` (~45 steps/epoch at
  batch 8 → ~26 epochs; augmentation on). T4 is plenty (~30–40 min); no need to chase an A100.

**Before uploading the zip** make sure the pool is current: re-run `promote_labels.py` +
`make_split.py` + `make_labeler_zip.sh` after any new adjudication (e.g. the 3 typo fixes).


In [ ]:
# Which GPU did we get? (T4 16GB is enough for this run)
!nvidia-smi


In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Copy the data package Drive -> VM disk and unzip (fast local disk for the dataloader).
%%time
!cp /content/drive/MyDrive/tnc/tnc_rung3_labeler_colab.zip /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/tnc_rung3_labeler_colab.zip
!ls /content/tnc/src/vision/ && head -c 300 /content/tnc/data/real/rung3/strips_r1/manifest.jsonl


In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless


In [ ]:
# Start weights: rung22-stemfix-best from Drive (left there by the Rung-2.2b run at
# MyDrive/tnc/rung22-stemfix/best). rsync skips the 1.1 GB trainer_state.pt (~550 MB copied).
# If you deleted it from Drive: re-upload data/checkpoints/rung22-stemfix-best/ (minus
# trainer_state.pt) to that path first.
%%time
!mkdir -p /content/rung22-stemfix-best
!rsync -a --exclude trainer_state.pt /content/drive/MyDrive/tnc/rung22-stemfix/best/ /content/rung22-stemfix-best/
!ls -la /content/rung22-stemfix-best


In [ ]:
# BASELINE on the real val BEFORE training — the number the labeler must beat.
# (rung22-stemfix has never seen these 56 strips; expect the synthetic->real gap here.)
%cd /content/tnc
!python src/vision/eval_omr.py --checkpoint /content/rung22-stemfix-best \
    --strips-dir data/real/rung3/strips_r1 --split data/real/rung3/strips_r1/split.json


In [ ]:
# SHAKEOUT (~2 min): 60 tiny steps to prove the wiring — expect `vocab: +0 tokens` (the start
# weights already carry the added tokens), augmentation workers, AMP, a checkpoint write to Drive,
# and falling loss. Run once per new setup.
%cd /content/tnc
!python src/vision/train.py --model /content/rung22-stemfix-best \
    --strips-dir data/real/rung3/strips_r1 --split data/real/rung3/strips_r1/split.json \
    --out-dir /content/drive/MyDrive/tnc/rung3-labeler-shakeout \
    --limit-train 128 --limit-val 56 --max-steps 60 --eval-every 30 --save-every 30 \
    --warmup-steps 10 --num-workers 2


In [ ]:
# THE RUN. Small LR, short schedule, eval every 100 steps (best-checkpoint selection on the
# held-out real pieces). T4 ~30-40 min.
%cd /content/tnc
!python src/vision/train.py --model /content/rung22-stemfix-best \
    --strips-dir data/real/rung3/strips_r1 --split data/real/rung3/strips_r1/split.json \
    --out-dir /content/drive/MyDrive/tnc/rung3-labeler \
    --lr 1e-5 --max-steps 1200 --warmup-steps 50 --eval-every 100 --save-every 100 \
    --num-workers 2


In [ ]:
# RESUME after a disconnect: re-run cells 1-5, then this (reloads model+optimizer+scheduler
# from .../rung3-labeler/last; --model is ignored on resume).
%cd /content/tnc
!python src/vision/train.py --model /content/rung22-stemfix-best \
    --strips-dir data/real/rung3/strips_r1 --split data/real/rung3/strips_r1/split.json \
    --out-dir /content/drive/MyDrive/tnc/rung3-labeler \
    --lr 1e-5 --max-steps 1200 --warmup-steps 50 --eval-every 100 --save-every 100 \
    --num-workers 2 --resume


In [ ]:
# AFTER: eval the best checkpoint on the same real val — compare against the baseline cell.
%cd /content/tnc
!python src/vision/eval_omr.py --checkpoint /content/drive/MyDrive/tnc/rung3-labeler/best \
    --strips-dir data/real/rung3/strips_r1 --split data/real/rung3/strips_r1/split.json


## After training (back on the Mac)

1. Download `MyDrive/tnc/rung3-labeler/best/` → `data/checkpoints/rung3-labeler-best/`.
2. Export ONLY what the emitter needs (it decodes via ONNX int8):
   `optimum-cli export onnx` → `src/vision/quantize_onnx.py` → `src/vision/onnx_parity.py`
   into `data/checkpoints/rung3-labeler-best-onnx/`.
3. **Do NOT ship it** — no `make_browser_gate.py`, nothing into `apps/web/public/models/`.
4. Sanity: `eval_omr.py --checkpoint data/checkpoints/rung3-labeler-best --strips-dir
   data/real/rung3/strips_r1 --split data/real/rung3/strips_r1/split.json` matches the Colab number,
   and take the frozen 33-strip exam via `eval_omr.py`'s per-source block (informational — the exam
   baseline 83.3% AEU belongs to the SHIPPED model, but a labeler that beats it on real strips is
   exactly what the emitter wants).
5. Point the emitter at it for every new emit (notaarsivleri, re-slice):
   `emit_strip_labels.py --checkpoint data/checkpoints/rung3-labeler-best
   --onnx-dir data/checkpoints/rung3-labeler-best-onnx --redecode ...`
6. Log the run in `src/vision/MODEL_EVAL.md` ("Rung 3 — Round-0.5 labeler").
